In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, roc_curve, auc, confusion_matrix, ConfusionMatrixDisplay

print("1. Cargando y preparando datos...")
df = pd.read_csv('dataset100k.csv') # Asegúrate de que el nombre coincida

# Feature Engineering
df = df.sort_values(by=['id_maquina', 'timestamp_lectura'])
df['delta_temp'] = df.groupby('id_maquina')['temp_c'].diff().fillna(0)
df['delta_vibracion'] = df.groupby('id_maquina')['vibracion_mms'].diff().fillna(0)
df['delta_corriente'] = df.groupby('id_maquina')['corriente_motor_a'].diff().fillna(0) # <- BUG CORREGIDO AQUÍ

features = ['rpm', 'vibracion_mms', 'temp_c', 'corriente_motor_a', 'delta_temp', 'delta_vibracion', 'delta_corriente']
X = df[features]
y = df['etiqueta_prediccion']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

print("2. Entrenando Modelo V3...")
pesos_clases = {0: 1, 1: 3}
rf_model = RandomForestClassifier(n_estimators=100, class_weight=pesos_clases, random_state=42, max_depth=12)
rf_model.fit(X_train, y_train)

# Guardamos el modelo para que lo descargues y uses en tu API
joblib.dump(rf_model, "modelo_cnc.pkl")

print("3. Generando Predicciones y Umbral 0.40...")
y_prob = rf_model.predict_proba(X_test)[:, 1]
umbral = 0.40
y_pred_ajustado = (y_prob >= umbral).astype(int)

print("\n--- 📊 MÉTRICAS PARA EL FRONTEND ---")
print(f"Accuracy: {accuracy_score(y_test, y_pred_ajustado):.4f}")
# Aquí sacamos el Recall de la clase 1 (Fallas)
from sklearn.metrics import recall_score, f1_score
print(f"Recall (Clase 1): {recall_score(y_test, y_pred_ajustado):.4f}")
print(f"F1-Score: {f1_score(y_test, y_pred_ajustado):.4f}")

print("\n4. Generando Imágenes para Streamlit...")

# --- GRAFICO 1: MATRIZ DE CONFUSIÓN ---
cm = confusion_matrix(y_test, y_pred_ajustado)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Normal", "Riesgo/Falla"])
disp.plot(cmap="Blues", values_format="d")
plt.title("Matriz de Confusión (Umbral 0.40)")
plt.savefig("matriz_confusion.png", bbox_inches='tight', dpi=300)
plt.close()

# --- GRAFICO 2: CURVA ROC ---
fpr, tpr, thresholds = roc_curve(y_test, y_prob)
roc_auc = auc(fpr, tpr)
plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (AUC = {roc_auc:.3f})')
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
plt.xlabel('Tasa de Falsos Positivos')
plt.ylabel('Tasa de Verdaderos Positivos')
plt.title('Curva ROC')
plt.legend(loc="lower right")
plt.savefig("curva_roc.png", bbox_inches='tight', dpi=300)
plt.close()

# --- GRAFICO 3: ÍNDICE GINI (Importancia) ---
importancias = rf_model.feature_importances_
plt.figure(figsize=(10, 6))
sns.barplot(x=importancias, y=features, palette="viridis")
plt.title("Importancia de las Variables (Índice Gini)")
plt.xlabel("Nivel de Importancia")
plt.ylabel("Sensores / Deltas")
plt.savefig("gini_importancia.png", bbox_inches='tight', dpi=300)
plt.close()

print("✅ ¡Listo! Descarga los archivos .png y el .pkl generados en la carpeta de Colab.")

1. Cargando y preparando datos...
2. Entrenando Modelo V3...
3. Generando Predicciones y Umbral 0.40...

--- 📊 MÉTRICAS PARA EL FRONTEND ---
Accuracy: 0.8760
Recall (Clase 1): 0.9426
F1-Score: 0.7576

4. Generando Imágenes para Streamlit...


/tmp/ipykernel_3668/1040985668.py:71: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `y` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(x=importancias, y=features, palette="viridis")


✅ ¡Listo! Descarga los archivos .png y el .pkl generados en la carpeta de Colab.
